In [5]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [6]:
# Fetch data
df=pd.read_csv("../spiff_data-2.csv")
mask=df==1000
df[mask]=pd.NA
df=df.dropna()
df=df.drop(columns=['Unnamed: 0', 'day'])

# Risk free rate
r = 0.03  

In [7]:
# Preliminary functions 
def log_returns(portfolie_value, period=1):
    maturity=len(portfolie_value)
    log_returns=np.log(portfolie_value[1:maturity]/portfolie_value[0:maturity-1])

    #annulize 
    log_returns=period*log_returns
    return log_returns
 
def SharpeRatio(log_return, r, period=1, low_volatilty_asset_adjustment=0):

    log_return = np.asarray(log_return)


    mu_hat = np.mean(log_return)
    sigma_hat = np.std(log_return)

    # Add a small epsilon if the asset and its returns have very low volatility (e.g money market account), to not divide by zero.
    # This matters in the grid search optimisation below, where if not added, the Sharp Ration blows up since the variance of the returns of 
    # a strategy with few trades tends to zero. 
    sigma_hat+=low_volatilty_asset_adjustment

    daily_SharpeRatio=(mu_hat - r/252) / sigma_hat

    # Return SharpeRatio "annualized" to the correct period. 
    return np.sqrt(period)*daily_SharpeRatio

In [8]:
# Channel breakout trade


def channel_breakout_trade(price_time_series,c_channel_width,x_trend_requirement,k_days_of_holding_stock,
                           time_span_for_comparison, sustained_trend_requirement,r,initial_portfolio_value,
                           allow_short_bool=False, allow_multiple_positions=True):
    count_low=0
    count_high=0
    maturity=len(price_time_series)

    money_account=np.zeros(maturity) # Money account, initially holding all portfolio value. 
    money_account[0]=initial_portfolio_value
   
    stock_position=np.zeros(maturity) # Store stock positions. 

    exit_position_signal=np.zeros(maturity) # Bookkeping for when to exit positions, -1 for selling (exiting long pos,), +1 for buying (exiting short pos.)


    for idx in range(1,maturity):
        # idx represents time

        money_account[idx]=money_account[idx-1] # Carry over money, adjust below if trades. 
        
        current_price=price_time_series[idx]

        if idx > time_span_for_comparison: # This is when we can start the channel trade. 
            stock_position[idx]=stock_position[idx-1] # Hold the same amount of stock, adjust below if enter/exit stock positions

            stock_price_high=max(price_time_series[idx-time_span_for_comparison-1:idx-1])
            stock_price_low=min(price_time_series[idx-time_span_for_comparison-1:idx-1])
            
            if stock_price_high<= (1+c_channel_width)*stock_price_low and stock_price_low >=(1-c_channel_width)*stock_price_high: # Check if we are in a price channel 

                upper_channel_bound=stock_price_low*(1+c_channel_width)
                lower_channel_bound=stock_price_high*(1-c_channel_width)

                # Case with a neative trend, potentially a short position scenario 
                if current_price <= (1 - x_trend_requirement)*lower_channel_bound and allow_short_bool: # Activates if shorting is allowed. 
                    count_low+=1
                    if count_low==sustained_trend_requirement:
                        if stock_position[idx]==0 or allow_multiple_positions:
                            stock_position[idx] -= 1 #Short stock, hold position for k days.
                            money_account[idx]+= current_price #Adjust money account when you short stock. 
                            count_low= 0 # set back count

                        if idx+k_days_of_holding_stock<maturity:
                            exit_position_signal[idx+k_days_of_holding_stock]=1 # Buy back stock in k days time. 
                else:
                    count_low=0 # Case with no trend, reset counting. 

                # Case with a positive trend, long position scenario. 
                if current_price >= (1+ x_trend_requirement)*upper_channel_bound:
                    count_high += 1
                    if count_high==sustained_trend_requirement:
                        if stock_position[idx]==0 or allow_multiple_positions:
                            stock_position[idx]+= 1 #Buy stock, hold position for k days.
                            money_account[idx] -= current_price #Adjust money account when you buy the stock. 
                            count_high=0

                        if idx+k_days_of_holding_stock<maturity:
                            exit_position_signal[idx+k_days_of_holding_stock]=-1 # Sell stock in k days time. 
                        count_high = 0 # set back count
                else:
                    count_high=0 # Case with no trend, reset counting. 

            else: # reset counting if there is no channel. 
                count_high=0
                count_low=0

        money_account[idx]=money_account[idx]*np.exp(r/252) # Moneyaccount grows at the risk free rate


        # Check if there are exit signals
        if exit_position_signal[idx]==1:
            stock_position[idx] += 1 # Close short. 
            money_account[idx] -= current_price #Adjust ZCB position when you buy the stock. 
        if exit_position_signal[idx]==-1:
            stock_position[idx] -= 1 # Close long. 
            money_account[idx] += current_price #Adjust ZCB position when you short stock. 


    
    # Liquidate any remaining position at maturity
    money_account[-1] += stock_position[-1] * price_time_series[-1]
    stock_position[-1] = 0

    portfolio_value=(stock_position*price_time_series + money_account)

    return [portfolio_value, stock_position, money_account]

In [9]:
# Gridsearch for tuning params. 
def channel_trade_grid_search_optimisation(time_series,r, allow_short_bool, allow_multiple_positions):
    c_channel_width=np.array([0.1, 0.5, 1, 5, 10])/100 #c
    x_trend_requirement=np.array([0.05, 0.1, 0.5, 1, 5])/100 #x
    k_days_of_holding_stock=np.array([1,2,3,5,10])
    time_span_for_comparison=np.array([5,10,15,20,25]) #j 
    sustained_trend_requirement=np.array([1,2,3]) #d 

    optimal_SharpeRatio=-10**10
    optimal_params=np.zeros(5)

    for c in c_channel_width:
        for x in x_trend_requirement:
            for k in k_days_of_holding_stock:
                for j in time_span_for_comparison:
                    for d in sustained_trend_requirement:
                    
                        portfolio_value=channel_breakout_trade(time_series,c,x,k,j,d,r,1,allow_short_bool, allow_multiple_positions)[0]
                        
                        returns=log_returns(portfolio_value)

                        # Optimise for high yearly Sharp Ratios, adding a volatility adjustment as to not favour params yielding few trades. 
                        low_volatility_adjustment=1
                        SharpeRatio_value=SharpeRatio(returns,r,252,low_volatility_adjustment)
    
                        if SharpeRatio_value > optimal_SharpeRatio:
                            optimal_SharpeRatio=SharpeRatio_value
                            optimal_params=np.array([c,x,k,j,d])
                            
    if np.array_equal(optimal_params, np.zeros(5)):
        print("Warning, no optimal params found in the gridsearch")

    return optimal_params

In [10]:
def channel_breakout_strategy_with_updated_optimised_params_and_shortened_trade_period(
    time_series,
    training_period,
    trade_period,
    r, 
    allow_short_bool,
    allow_multiple_positions
):    

    maturity = len(time_series)

    # --- GLOBAL STORAGE ---
    full_stock_position = []
    full_money_account = []
    full_prices = []

    current_portfolio_value = 1.0

    # move forward by trade_period each iteration
    for start in range(
        0,
        maturity - training_period,
        trade_period
    ):

        # Fetch training prices
        train_start = start
        train_end = start + training_period
        training_prices = time_series[train_start:train_end]

        # optimise parameters on training window
        params = channel_trade_grid_search_optimisation(
            training_prices,
            r, 
            allow_short_bool,
            allow_multiple_positions
        )

        c = params[0]
        x = params[1]
        k = int(params[2])
        j = int(params[3])
        d = int(params[4])

        # Fetch trading prices
        trade_start = train_end
        trade_end = min(trade_start + trade_period, maturity)

        trading_prices = time_series[trade_start:trade_end]
        full_prices.append(trading_prices)
        

        # Trade! 
        portfolio_value, stock_position, money_account = (
            channel_breakout_trade(
                trading_prices,
                c,
                x,
                k,
                j,
                d,
                r,
                current_portfolio_value, 
                allow_short_bool,
                allow_multiple_positions
            )
        )

        # Store trade 
        full_stock_position.append(stock_position)
        full_money_account.append(money_account)

        # Liquidify position, start next trading round holding the current value in the money market account. 
        current_portfolio_value = portfolio_value[-1]


    # Collect results
    full_stock_position = np.concatenate(full_stock_position)
    full_money_account = np.concatenate(full_money_account)
    full_prices=np.concatenate(full_prices)

    # Get total portfolio value. 
    portfolio_value = (
        full_stock_position * full_prices
        + full_money_account
    )

    return (
        portfolio_value,
        full_stock_position,
        full_money_account
    )


In [ ]:
# Perform the trading scheme

asset_name="gurkor" # Chose asset to trade on. 
price_series = df[asset_name].values.astype(float)

training_period = 500 # How big data sets we train on. 
trade_period = 50 # Amount of drading days for a given set of tuned params. 
trade_horizon=4000 # Time horizon of the trading scheme, i.e we trade up to day numebr 'trade_horizon'. 
allow_short_bool = False # Allow short positions or not. 
allow_multiple_positions = False # Allow multiple positions, i.e. owning more than one unit of stock at the time. 

# Trade! 
portfolio_value, stock_position, money_account = (
        channel_breakout_strategy_with_updated_optimised_params_and_shortened_trade_period(
            time_series=price_series[:trade_horizon],
            training_period=training_period,
            trade_period=trade_period,
            r=r,
            allow_short_bool=allow_short_bool,
            allow_multiple_positions=allow_multiple_positions
        )
    )

# --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** ---
# Get returns of the scheme and of holding the risk-free asset. 
strategy_returns = log_returns(portfolio_value,252)

    # risk free asset value
maturity = len(portfolio_value)
risk_free_asset = np.exp(
        (r / 252) * np.arange(1,maturity+1)
    )
    # risk free log returns
rf_returns = log_returns(risk_free_asset, period=252)

sharpe_ratio = SharpeRatio(
        strategy_returns,
        r=r,
        period=252,
    )

# --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** --- *** ---
#%%
# Plot everything. 

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(3, 1, height_ratios=[1, 3, 3])

ax0 = fig.add_subplot(gs[0])
ax0.axis('off')
sharpe_text = (
    f"Using data of the {trade_horizon} first days."
    f"Training window: {training_period} days. Trading window: {trade_period} days. \n"
    f"Allowing for holding multiple stocks: {allow_multiple_positions}. "
    f"Allow shorting: {allow_short_bool}. Sharpe ratio (annualized): {sharpe_ratio:.4f}"
)
ax0.text(
        0.5,
        0.5,
        sharpe_text,
        fontsize=16,
        ha='center',
        va='center',
        bbox=dict(
            facecolor='lightblue',
            edgecolor='black',
            boxstyle='round,pad=0.5'
        )
    )
ax0.set_title(
        f"Channel breakout scheme, asset: {asset_name}",
        fontsize=18,

    )

# Upper plot, portfolio value. 
ax1 = fig.add_subplot(gs[1])
ax1.plot(
        portfolio_value,
        label='Strategy Portfolio Value',
        linewidth=2
    )
ax1.plot(
        risk_free_asset,
        label='Risk Free Asset',
        linestyle='--',
        linewidth=2
    )
ax1.set_title("Portfolio Value vs Risk Free Asset")
ax1.set_xlabel("Time")
ax1.set_ylabel("Value")
ax1.legend()
ax1.grid(True)

# Lower plot, returns. 
ax2 = fig.add_subplot(gs[2])
ax2.plot(
        strategy_returns,
        label='Strategy Log Returns',
        linewidth=1
    )
ax2.plot(
        rf_returns,
        label='Risk Free Return',
        linestyle='--',
        linewidth=2
    )
ax2.set_title("Strategy Returns vs Risk Free Return")
ax2.set_xlabel("Time")
ax2.set_ylabel("Log Return")
ax2.legend()
ax2.grid(True)

# Final lines 
plt.tight_layout()
plt.show()





/var/folders/0x/zd5rqg451_530rdy48v7l4b80000gn/T/ipykernel_43834/393990014.py:4: RuntimeWarning: invalid value encountered in log
  log_returns=np.log(portfolie_value[1:maturity]/portfolie_value[0:maturity-1])


KeyboardInterrupt: 